---
title: "More with XML: MODS and lxml"
---

The [Metadata Object Description Schema](https://www.loc.gov/standards/mods/), 
more commonly known by its acronym MODS, is a frequently used metadata
schema used by libraries and other cultural heritage organizations to record and transfer information
about digital objects, bibliographic items, and other sorts of collection materials.
While DublinCore was designed for describing any kind of digital content,
MODS was specifically intended to be a web-friendly scheme for bibliograpic data, particularly the MaRC standard (see @miller2022 [pg. TODO]). While MODS is much simpler than the MaRC structure, it is 
more complex (and possibly less widely used) than the DublinCore schema.
However, you will frequently encounter MODS when working
with digital content maintained by libraries and archives, since
it is designed to be harmonious with the most widely used metadata approaches used in these types of organizations.

## Parsing with `lxml`

Although the ElementTree module is a standard Python module for processing XML, there are 
other options for parsing XML in Python. The [lxml](https://lxml.de/) library provides 
greater support for XPath, provides a more Element-focused library (vs ET's focus on the tree structure).
Beyond the basics, it also supports efficient XML writing, and provides support for XML validation.

### Setup

First, import the `lxml` library, ensure it's running.

In [ ]:
from lxml import etree

Then, load the data. 

### Loading MODS Data

If you use the cells below that use LXML on its own, then you'll also need to import
the library so you can call it directly:

In [ ]:
try:
    from lxml import etree
    print('running with lxml.etree as etree')
except ImportError:
    print('you\'re not running with lxml active')

Now set up the data and other imports.
The cell below loads the data and prepares and output path for any write activites.
Later activies also use regular expressions to validate strings against certain patterns.
using the `re` module for regular expressions.
Also call the `pathlib` for working with file paths.

In [ ]:
import re
from pathlib import Path

DATA = Path('..', 'data')
OUT = Path('..', 'output')

## First Step: Load the records

Five MODS sample records are located in the `../data` directory. The next cell assigns the five sample records to the `MODS_collection` variable:

In [ ]:
MODS_collection = DATA / 'lcwa-mods-5.xml'

Next, load and parse the records to look through the contents. This
requires some advance knowledge of the tags and attributes you would expect to find in the record,
which you can find in the documentation linked above.

In [ ]:
tree = etree.parse(MODS_collection)
root = tree.getroot()

You can use a for loop to see what top-level elements are in the tree:

In [ ]:
for item in root:
    print(item.tag)

:::{tip} Count the elements in a collection of metadata
To get an idea of the scope of elements in collection of 5 MODS records, you can use the `.iter()` function.

The xml library's `.iter()` function produces a reusable
data object that you can iterate through using a `for` loop.
Here's some starter code that would list out the elements in your loaded metadata:

```python
for element in root.iter():
    print(element.tag)
```

How would you add a counter?
:::

Unhide the cell below to see how you could implement a counter in the code:

In [ ]:
el_count = 0 

for element in root.iter():
    el_count += 1
    print(element.tag)

print(el_count)

```{exercise} Use python to count the XML elements
:label: xml-count-elements
While it isn't usually necessary to know how many elements are in your metadata, it can be a good measure of record completeness. It is not easy to count these by hand, so this is a great task for automation tools!

How many elements did you find in the collection of 5 MODS records?
```

:::{solution} xml-count-elements
:label: xml-count-elements-solution
:class: dropdown
There are **247** elements in the collection of 5 MODS records.
:::

### Using BeautifulSoup

The following cells parse a record using BeautifulSoup and lxml together:

:::{note} TODO: Heading and intro
Update **Using BeautifulSoup** and "parse a record using BeautifulSoup and lxml together." This section now parses with `lxml` alone (`etree.parse()` on the open file, then `.getroot()`).
:::

In [ ]:
fhand = open(MODS_collection, 'rb')

tree = etree.parse(fhand)
root = tree.getroot()

In [ ]:
print(''.join(root.itertext())[:100])

The above cell prints a string of text from the XML object that we've loaded, 
in order to demonstrate that, yes, we have loaded content. 

Later we will pull more meaningful information. For now, let's quickly 
pull out some of the titles, this time using the tag names: 

In [ ]:
for tag in root.iter('{*}title'):
    print(etree.QName(tag).localname, tag.text)

:::{note} TODO: Functions that need explanation
New functions and syntax to introduce in the text: `root.itertext()` (all text in the tree, standing in for `soup.text`), `root.iter('{*}title')` (iterate by tag name; `{*}` matches any namespace, so this works before the Namespaces section), and `etree.QName(tag).localname` (tag name without the `{namespace}` prefix, standing in for `tag.name`). Also: `tag.text` is only the element's own text, not the text of its children as in BeautifulSoup. The text before **Navigating and Exploring the Tree** ("this time using the tag names") should be revisited with this in mind.
:::

### Navigating and Exploring the Tree

To get a list of all the tags in the document, try something like this (using `True` to demonstrate the existence of each tag in the file). Here, note the use of the `limit` argument to return only 10 instances. We don't need the whole list here for the purposes of demonstration:

In [ ]:
count = 0

for tag in root.iter('*'):
    print(etree.QName(tag).localname)
    count += 1
    if count >= 20:
        break

:::{note} TODO: Update `True` and `limit` text
The text before this cell describes `find_all(True, ...)` and the `limit` argument. These are BeautifulSoup features; the lxml version uses `root.iter('*')` with a counter and `break`.
:::

Or, we could look for each of the attributes on the top-level `mods` tags. We can see that 
they are stored in a dictionary-like object, which is indicated with curly braces `{}`):

In [ ]:
for tag in root.iterfind('.//{*}mods'):
    print(etree.QName(tag).localname, dict(tag.attrib))

:::{note} TODO: Update attributes text
The text before this cell says attributes are "stored in a dictionary-like object." In lxml this is `tag.attrib`, converted here with `dict()` for printing.
:::

Finally, don't forget to close the file:

In [14]:
fhand.close()

:::{note} TODO: Closing the file
The text before this cell ("don't forget to close the file") still applies as written because the file is opened with `open()`. `etree.parse()` also accepts a path directly (as the later cells do), which would make this step, and the cell above it, unnecessary.
:::

### Namespaces

In some cases, you may be working with data that has tags from various namespaces (that is, 
basically different tag schemas, such as MODS or EAD). For specificity, it can be 
important to have a list of the namespaces that you will reference. In this case, 
that list is a python dictionary named `ns`: 

In [15]:
ns = {
    'mods' : 'http://www.loc.gov/mods/v3',
    'ead3'  : 'http://ead3.archivists.org/schema/',
}

:::{note} TODO: Namespaces placement
The `ns` dictionary is defined here, after the cells above that needed to avoid namespaces by using `{*}`. Consider moving this section before **Using BeautifulSoup** (now lxml), or explaining the `{*}` wildcard here. The `ead3` entry is not used anywhere in the notebook.
:::

## First Steps to Navigate the Tree Using LXML / XPath

Load records, list tags and child elements, see subelements, display tags and attributes... 

## Examples

Here are some things we'll do using the BeautifulSoup library,
to be developed below in the notebook following:
 
1. Looking in a single XML file with multiple `<mods>` subelements, use a loop to count the elements. 
1. Using a similar loop to the above, can you go through the individual `<mods>` and extract the record's identifier?  
1. Let's get a bit deeper into MODS. These records use the `<titleInfo>` and `<title>` tags. Use the `.findall()` function to look into each of these and pull out the text of each `<title>` element. 
1. The above pulls out all the titles, including related titles. Use the `.find()` funtion to search for just the first instance and pull out only the main titles.  
1. These records contain `<subject>` designations, but only some of these correspond to headings that are authorized headings in the Library of Congress Subject Headings. Those are marked with an attribute `authority='lcsh'`, which is indicated as an embedded attribute in the tag. Look through `<subject>` tags, identify only the ones that include an LCSH attribute, then print the content of those subject headings.  
1. Data addition or modification: identify the local call numbers, then check to make sure all of them have appropriate attribute data attached.
1. Data validation: check the local call number references to ensure that they are in the proper format (e.g., _lcwaAddddddd_).
1. Save the updated metadata. In this case, write the udpated metadata to a new file.

:::{note} TODO: Examples list
The intro line "things we'll do using the BeautifulSoup library" needs updating. Items 3 and 4 mention `.findall()` and `.find()`, which are now accurate for lxml.
:::

In [ ]:
MODS_file = DATA / 'lcwa-mods-5.xml'

#### 1. Counting Records in the Set

Activity 1: How many metadata records for discrete items are included in the set? A compound MODS file may create multiple records in one file; in this scenario, the tag `<mods>` encloses each individual record, and the list of records is enclosed in a `<modsCollection>` tag. Use a loop to count the `<mods>` elements.

In [ ]:
#lxml
record_count = 0

xml_records = etree.parse(MODS_file)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    print(etree.QName(mods).localname, mods.findtext('.//mods:title', namespaces=ns), mods.findtext('.//mods:form', namespaces=ns))
    record_count += 1

print(record_count)

#### 2. Extract Item Identifiers

Activity 2: Each individual metadata record has at least one `<identifier>` element; this element is used to include a reference to the item, such as a URI, or another identifier that a system may use to locate an item. Using a loop similar to the example above, how would you print each record's identifier(s)?  

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier', namespaces=ns):
        print(etree.QName(identifier).localname, identifier.text)

There are clearly different types of identifiers here, and when we check the identifier attributes, 
it is clear that some of these will be more useful than others. Below, use the `.attrs` method to see the dictionary that each element carries:

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier', namespaces=ns):
        print(etree.QName(identifier).localname, identifier.text, dict(identifier.attrib))

Since some of the elements do not have attributes, we need a try-except loop to look at each dictionary, 
and to generate a "Blank" value for elements without attributes:

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier', namespaces=ns):
        tag = etree.QName(identifier).localname
        content = identifier.text
        try:
            type_ = identifier.attrib['type']
        except KeyError:
            type_ = "Blank type"
        print(tag, content, type_)

Finally, let's print a clean list of only the URI identifiers:

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier[@type="uri"]', namespaces=ns):
        print(identifier.attrib['type'], identifier.text)

:::{note} TODO: Activity 1 and 2 text
Activity 1 prints `mods.title` and `mods.form` as `findtext()` values rather than tag markup. The text before the `.attrs` cell ("use the `.attrs` method") should now say `.attrib`. The text before the "Blank type" cell can mention `.get('type', 'Blank type')` as a shorter alternative to `try`/`except`. In the final cell, the BeautifulSoup `type="uri"` keyword filter is now the XPath predicate `[@type="uri"]`.
:::

Now, we could try that another way using the `lxml` XML library directly:

In [22]:
#lxml
xml_records = etree.parse(MODS_file)

for identifier in xml_records.findall('.//mods:identifier', namespaces=ns): 
    element = identifier
    print(element.tag, element.text, element.attrib)

{http://www.loc.gov/mods/v3}identifier lcwaN0010234 {}
{http://www.loc.gov/mods/v3}identifier 85999 {'invalid': 'yes', 'type': 'database id'}
{http://www.loc.gov/mods/v3}identifier 109353 {'invalid': 'yes', 'type': 'database id'}
{http://www.loc.gov/mods/v3}identifier http://www.slate.com/ {'displayLabel': 'Access URL', 'type': 'uri'}
{http://www.loc.gov/mods/v3}identifier 15046 {'type': 'database id'}
{http://www.loc.gov/mods/v3}identifier lcwaN0001999 {}
{http://www.loc.gov/mods/v3}identifier 91224 {'invalid': 'yes', 'type': 'database id'}
{http://www.loc.gov/mods/v3}identifier 109272 {'invalid': 'yes', 'type': 'database id'}
{http://www.loc.gov/mods/v3}identifier http://rawstory.com/ {'displayLabel': 'Access URL', 'type': 'uri'}
{http://www.loc.gov/mods/v3}identifier 2771 {'type': 'database id'}
{http://www.loc.gov/mods/v3}identifier lcwaN0003238 {}
{http://www.loc.gov/mods/v3}identifier 91275 {'invalid': 'yes', 'type': 'database id'}
{http://www.loc.gov/mods/v3}identifier 109273 {'

And, filter to identify only the URI elements...

Using an XPath expression `etree`, you can filter directly for identifiers that
have a `type` attribute with the `uri` value:

In [23]:
#lxml
xml_records = etree.parse(MODS_file)

for identifier in xml_records.findall('.//mods:identifier[@type="uri"]', namespaces=ns): 
    element = identifier
    attribs = element.attrib
    type = attribs.get('type')
    print(element.tag, type, element.text)

{http://www.loc.gov/mods/v3}identifier uri http://www.slate.com/
{http://www.loc.gov/mods/v3}identifier uri http://rawstory.com/
{http://www.loc.gov/mods/v3}identifier uri http://www.huffingtonpost.com/
{http://www.loc.gov/mods/v3}identifier uri https://medium.com/buzzfeed-collections
{http://www.loc.gov/mods/v3}identifier uri http://www.buzzfeed.com/
{http://www.loc.gov/mods/v3}identifier uri http://www.drudgereport.com/


:::{note} TODO: Redundant cells in Activity 2
The cells above now give the same results as the earlier lxml cells here. The text before them ("we could try that another way using the `lxml` XML library directly" and "Using an XPath expression `etree`") no longer describes a second approach, so it needs rewriting, or the duplicate cells need to be consolidated.
:::

#### 3. Extract the record titles 

Activity 3: Each individual metadata record has `<titleInfo>` that contains at least one `<title>` element; this element is used to identyify an item's title. Using a loop similar to the example above, how would you print each record's title(s)?  

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for title in mods.findall('.//mods:title', namespaces=ns):
        print(etree.QName(title).localname, etree.QName(title.getparent()).localname)

:::{note} TODO: Activity 3 text
This cell now prints the parent's tag name rather than the parent's full markup (BeautifulSoup's `title.find_parent()` printed the whole `titleInfo` element).
:::

Similarly, `lxml` may be used with the `.getparent()` function. Note that this is not possible with the ElementTree module, only with the lxml module.

In [25]:
#lxml
xml_records = etree.parse(MODS_file)
metadata = xml_records.getroot()

for mods in metadata.findall('.//mods:mods', namespaces=ns):
    for title in mods.findall('.//mods:title', namespaces=ns):
        parent = title.getparent()
        print(title.tag, parent.tag)

{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}titleInfo
{http://www.loc.gov/mods/v3}title {http://www.loc.gov/mods/v3}ti

Note above that the `.find_parent()` (BS4) or `.getparent()` (lxml) methods can be used to look "up" the tree, 
in this case displaying the parent element of the `title` element.

A similar result can be produced using the `lxml` library directly. In this case, 
the `.findall()` method is similar, but notice that the request can be given
using XPATH references and while specifying namespaces:

:::{note} TODO: Activity 3 text
The text before the first Activity 3 cell and the text above ("the `.find_parent()` (BS4) or `.getparent()` (lxml) methods..." and "Similarly, `lxml` may be used with `.getparent()`... not possible with the ElementTree module") needs updating, since both cells are now lxml. The cell after this text is redundant with the cell before it, and the cell following that one (`for title in xml_records.findall...`) duplicates it.
:::

In [26]:
#lxml
xml_records = etree.parse(MODS_file)

for titleInfo in xml_records.findall('.//mods:title', namespaces=ns): 
    element = titleInfo
    print(element.text)

Slate Magazine
General News on the Internet Web Archive
Serial and Government Publications Division
Raw Story
General News on the Internet Web Archive
Serial and Government Publications Division
Huffington Post
General News on the Internet Web Archive
Serial and Government Publications Division
BuzzFeed
General News on the Internet Web Archive
Serial and Government Publications Division
Drudge Report
General News on the Internet Web Archive
Serial and Government Publications Division


In [27]:
for title in xml_records.findall('.//mods:title', namespaces=ns):
    print(title.text)

Slate Magazine
General News on the Internet Web Archive
Serial and Government Publications Division
Raw Story
General News on the Internet Web Archive
Serial and Government Publications Division
Huffington Post
General News on the Internet Web Archive
Serial and Government Publications Division
BuzzFeed
General News on the Internet Web Archive
Serial and Government Publications Division
Drudge Report
General News on the Internet Web Archive
Serial and Government Publications Division


#### 4. Extract only the Main Titles

Activity 4: Notice in the previous activity that even though we are working with only five
records, there are well more than five titles. Each of these records has multiple `title` elements,
some of which are for `relatedItem` elements. If we want only the main titles, use the `.find()` function 
to search for only the first instance and print out only the main title. An alternative way to do this in `lxml`
is to use a more specific XPath selector.

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    title = mods.find('.//mods:title', namespaces=ns)
    print(etree.QName(title).localname, title.text)

:::{note} TODO: Activity 4 text
The text before this cell says to use the `.find()` function, which is what this cell now does with lxml. The cell after it ("Try `lxml` to use an XPath request... you can also use the `.find()` method in `lxml`") needs its text updated, since `.find()` is no longer new there.
:::

Try `lxml` to use an XPath request. (Note: you can also use the `.find()` method in `lxml` to return only the first result.)

In [29]:
#lxml
xml_records = etree.parse(MODS_file)
metadata = xml_records.getroot()

for title in metadata.findall('.//mods:mods/mods:titleInfo/mods:title', namespaces=ns): 
    print(title.text, title.tag)

Slate Magazine {http://www.loc.gov/mods/v3}title
Raw Story {http://www.loc.gov/mods/v3}title
Huffington Post {http://www.loc.gov/mods/v3}title
BuzzFeed {http://www.loc.gov/mods/v3}title
Drudge Report {http://www.loc.gov/mods/v3}title


Above, the query specifically asks for the `title` elements that are direct 
child elements of a `titleInfo` element, which is a child of the `mods` element. 
This is necessary to filter out any `titleInfo` elements that are actually under a
`relatedItem` element. With less specificity, the query will return numerous elements 
that "related" but not the title of the actual item: 

#### 5. Exploring the Subject Element

Activity 5: These records contain `<subject>` designations, but only some of these correspond to headings that are authorized headings in the Library of Congress Subject Headings (LCSH). Those are indicated with an attribute `authority='lcsh'`, which is indicated as an embedded attribute in the tag. Look through the `<subject>` tags and identify only the ones that include an LCSH attribute, then print the content of those subject headings.

Note: this activity requires using the twenty-five record set rather than the one with five records.

As LCSH headings are generally constructed as a main topic word, followed by descriptors that indicate further
topical, geographic, or chronological details, note that the structure is mimicked here, with `<topic>` and
various `<genre>`, `<geographic>`, or other specifiers.

In [ ]:
large_MODS_collec = DATA / 'lcwa-mods-25.xml'

First, take a look at the structure of the subject element sections using the `.tostring()` options:

In [31]:
#lxml
xml_records = etree.parse(large_MODS_collec)
metadata = xml_records.getroot()

for subject in metadata.findall('.//mods:subject[@authority="lcsh"]', namespaces=ns):
#   uncomment the next line if you want to see the tag names and attributes
#    print(subject.tag, subject.attrib)
    print(etree.tostring(subject, encoding='utf-8').decode('utf-8'))

<subject xmlns="http://www.loc.gov/mods/v3" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" authority="lcsh">
        <name authority="naf" type="corporate">
            <namePart><!-- TODO: Insert name authority here (can be same as name authority above, under title). --></namePart>
        </name>
    </subject>
    
<subject xmlns="http://www.loc.gov/mods/v3" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" authority="lcsh">
        <topic>Animals</topic>
        <genre>Pictorial works</genre>
    </subject>
    
<subject xmlns="http://www.loc.gov/mods/v3" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" authority="lcsh">
        <name authority="naf" type="corporate">
            <namePart><!-- TODO: Insert name authority here (can be same as name authority above, under title). --></namePart>
        </name>
    </subject>
    
<subject xm

In [32]:
#lxml
xml_records = etree.parse(large_MODS_collec)
metadata = xml_records.getroot()

for subject in metadata.findall('.//mods:subject[@authority="lcsh"]', namespaces=ns):
    print(subject.tag, subject.attrib)
    for item in subject:
        print(item.tag, item.text)
    print('\n')

{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}name 
            


{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}topic Animals
{http://www.loc.gov/mods/v3}genre Pictorial works


{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}name 
            


{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}topic Memes


{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}name 
            


{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}topic Memes


{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}name 
            


{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}topic Memes


{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}name 
            


{http://www.l

In [ ]:
#lxml
xml_records = etree.parse(large_MODS_collec)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for subject in mods.findall('.//mods:subject[@authority="lcsh"]', namespaces=ns):
        print(etree.tostring(subject, encoding='unicode', with_tail=False), dict(subject.attrib), '\n')

In [ ]:
# demonstrating navigating down the tree from the subject element to its first topic
#lxml
xml_records = etree.parse(large_MODS_collec)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for subject in mods.findall('.//mods:subject[@authority="lcsh"]', namespaces=ns):
        print(subject.findtext('mods:topic', namespaces=ns))

:::{note} TODO: Activity 5 text
The two BeautifulSoup cells (`print(subject, subject.attrs)` and the dot-notation `subject.topic` cell) are now lxml. The dot-notation comment was reworded since lxml has no dot notation; this cell now uses `findtext('mods:topic')`. The text for the next cells ("Using `lxml`, we can use XPath queries") can be revised, since the previous cells already use lxml. Both cells would also be candidates for slicing the output (`count`/`break`) if the length is a concern.
:::

Using `lxml`, we can use XPath queries. (Again, remember to use the 25-record file, not the 5-record file.)

In [36]:
#lxml
xml_records = etree.parse(large_MODS_collec)

for subject in xml_records.findall('.//mods:subject', namespaces=ns): 
    print(subject.tag, subject.attrib)

{http://www.loc.gov/mods/v3}subject {'authority': 'keyword'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcwabt'}
{http://www.loc.gov/mods/v3}subject {'authority': 'keyword'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcwabt'}
{http://www.loc.gov/mods/v3}subject {'authority': 'keyword'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcwabt'}
{http://www.loc.gov/mods/v3}subject {'authority': 'keyword'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject {'authority': 'lcwabt'}
{http://www.loc.gov/mods/v3}subject {'authority': 'k

Similarly, we can filter to view only those with `lcsh` subject authorities:

In [37]:
#lxml
xml_records = etree.parse(large_MODS_collec)

for subject in xml_records.findall('.//mods:subject[@authority="lcsh"]', namespaces=ns): 
    print(subject.tag, len(subject), subject.attrib)

{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 2 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 2 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 2 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 2 {'authority': 'lcsh'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh', 'displayLabel': 'United States Elections, 2014'}
{http://www.loc.gov/mods/v3}subject 1 {'authority': 'lcsh', 'displayLabel': 'United States Elections, 2014'}
{h

:::{note} TODO: Activity 5 contrast passage
The passage beginning "You may notice that the lxml tools are more literal..." contrasts lxml with BeautifulSoup ("unlike using BeautifulSoup when we can ask for the contents of a tag"), but no BeautifulSoup code remains in the notebook to illustrate it. Needs rewriting, or a BeautifulSoup example retained for the contrast.
:::

You may notice that the lxml tools are more literal, in a sense, 
meaning that they really only usually give you exactly what you ask for. 
So, for example, unlike using BeautifulSoup when we can ask for the contents 
of a tag (that is, all of the text that is enclosed by the tag), 
`lxml` treats the metadata more like data. In this case, the `subject` tags don't
strictly contain any actual text (that is, nothing the parser recognizes as 
a string of characters), in fact they only contain more sublements, which then contain text. 
The structure looks something like this: 

```xml
<subject authority="lcsh">
    <topic>Animals</topic>
    <genre>Pictorial works</genre>
</subject>
```
So, as the `lxml` parser sees it, the "contents" of the `<subject>` tag 
are two subelements: `<topic>` and `<genre>`. 
To get the "text" or content of the element, we need to look at the attributes
(in this case, the authority type), and then to extract the subelements. 
Only when the subelements are obtained can we retrieve their text.

First, look for the contents of the `subject` element, the list of its subelements,
here identified as the "child" elements since they are further "down" in the tree:

In [38]:
#lxml
xml_records = etree.parse(large_MODS_collec)
count = 0 

for subject in xml_records.findall('.//mods:mods/mods:subject', namespaces=ns): 
    if subject.attrib['authority'] == 'lcsh':
        count += 1
        print(subject.tag, count, 'children:')
        for subelement in subject:
            print('  ',subelement.tag)
        print('\n')
        if count > 4:
            break

{http://www.loc.gov/mods/v3}subject 1 children:
   {http://www.loc.gov/mods/v3}name


{http://www.loc.gov/mods/v3}subject 2 children:
   {http://www.loc.gov/mods/v3}topic
   {http://www.loc.gov/mods/v3}genre


{http://www.loc.gov/mods/v3}subject 3 children:
   {http://www.loc.gov/mods/v3}name


{http://www.loc.gov/mods/v3}subject 4 children:
   {http://www.loc.gov/mods/v3}topic


{http://www.loc.gov/mods/v3}subject 5 children:
   {http://www.loc.gov/mods/v3}name




Finally, to extract the actual subject terms, we can request the text of the subelements:

In [39]:
#lxml
xml_records = etree.parse(large_MODS_collec)
count = 0

for subject in xml_records.findall('.//mods:mods/mods:subject', namespaces=ns): 
    if subject.attrib['authority'] == 'lcsh':
        count += 1
        print(subject.tag, count, 'children:')
        for subelement in subject:
            print(f'  {subelement.tag.split("}")[1]} - {subelement.text}')
        print('\n')
        if count > 4:
            break

{http://www.loc.gov/mods/v3}subject 1 children:
  name - 
            


{http://www.loc.gov/mods/v3}subject 2 children:
  topic - Animals
  genre - Pictorial works


{http://www.loc.gov/mods/v3}subject 3 children:
  name - 
            


{http://www.loc.gov/mods/v3}subject 4 children:
  topic - Memes


{http://www.loc.gov/mods/v3}subject 5 children:
  name - 
            




In [40]:
#lxml
xml_records = etree.parse(large_MODS_collec)
count = 0

for subject in xml_records.findall('.//mods:subject[@authority="lcsh"]', namespaces=ns):
    count += 1
    print(subject.tag, count, 'children:')
    print(f'  type: {subject.attrib["authority"]}')
    for subelement in subject:
        print(f'  {subelement.tag.split("}")[1]} - {subelement.text}')
    print('\n')
    if count > 4:
        break

{http://www.loc.gov/mods/v3}subject 1 children:
  type: lcsh
  name - 
            


{http://www.loc.gov/mods/v3}subject 2 children:
  type: lcsh
  topic - Animals
  genre - Pictorial works


{http://www.loc.gov/mods/v3}subject 3 children:
  type: lcsh
  name - 
            


{http://www.loc.gov/mods/v3}subject 4 children:
  type: lcsh
  topic - Memes


{http://www.loc.gov/mods/v3}subject 5 children:
  type: lcsh
  name - 
            




Try the above and look for other "authority" types in the list of records.

#### 6. Data Addition or Modification

Activity 6: Now that we can request things in the tree, let's look for 
more specific things, like content strings that meet certain criteria, 
then add or modify content to enhance them. 

Let's return to the `identifier` elements. Some of these are structured with
local call numbers, but those don't appear to be identified with any additional 
attributes:

```xml
  <identifier>lcwaN0010234</identifier>
  <identifier invalid="yes" type="database id">85999</identifier>
  <identifier invalid="yes" type="database id">109353</identifier>

```

Would it be possible to modify these and add a "type" attribute for those
local numbers? Let's start with BeautifulSoup:

:::{note} TODO: Activity 6 text
The text above ends with "Let's start with BeautifulSoup:" and needs updating. The next cells (show identifiers, regex match, add attributes, check as XML) are now lxml: attributes are set with `identifier.set(name, value)` instead of `identifier['type'] = ...`, and the check cell uses `etree.tostring(..., pretty_print=True)` instead of `.prettify()`. The regex explanation still applies.
:::

In [ ]:
#lxml - show the identifiers and their attributes
xml_records = etree.parse(large_MODS_collec)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier', namespaces=ns):
        print(etree.QName(identifier).localname, identifier.text, dict(identifier.attrib))

In [ ]:
#lxml - use regular expressions to identify the local identifiers or "call numbers"

# set up a regex pattern
call_num_pattern = re.compile(r'[a-z]{4}N\d{7}')
# alternatively, be more specific and look for the lcwa string at the beginning:
# call_num_pattern = re.compile(r'^blcwaN\d{7}')

xml_records = etree.parse(large_MODS_collec)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier', namespaces=ns):
        if re.match(call_num_pattern, identifier.text):
            print(etree.QName(identifier).localname, identifier.text, dict(identifier.attrib))

In [ ]:
#lxml

# now, add in new attributes for these "local call number" elements
#lxml - use regular expressions to identify the local identifiers or "call numbers"

# set up a regex pattern
call_num_pattern = re.compile(r'[a-z]{4}N\d{7}')
# alternatively, be more specific and look for the lcwa string at the beginning:
# call_num_pattern = re.compile(r'^blcwaN\d{7}')

xml_records = etree.parse(large_MODS_collec)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier', namespaces=ns):
        if re.match(call_num_pattern, identifier.text):
            # add attributes by assigning values
            identifier.set('type', 'local_call_number')
            identifier.set('invalid', 'no')
            identifier.set('displaylabel', 'Local Call Number')
            identifier.set('updated', 'true')
            #print, to make sure that these were added
            print(etree.QName(identifier).localname, identifier.text, dict(identifier.attrib))

In [ ]:
#lxml - check to make sure they look okay as XML

# set up regex
call_num_pattern = re.compile(r'[a-z]{4}N\d{7}')

xml_records = etree.parse(large_MODS_collec)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier', namespaces=ns):
        if re.match(call_num_pattern, identifier.text):
            # add attributes by assigning values
            identifier.set('type', 'local_call_number')
            identifier.set('invalid', 'no')
            identifier.set('displaylabel', 'Local Call Number')
            print(etree.tostring(identifier, pretty_print=True, encoding='unicode', with_tail=False))

:::{note} TODO: Activity 6 duplication
The BeautifulSoup and lxml versions of Activity 6 are now both lxml. The cells after this one ("#lxml identify the local call number identifiers" and "#lxml insert attributes...") repeat the same steps and could be consolidated with the cells above.
:::

In [45]:
#lxml identify the local call number identifiers

xml_records = etree.parse(large_MODS_collec)

# regex pattern to identify the call number:
call_num_pattern = re.compile(r'[a-z]{4}N\d{7}')

for identifier in xml_records.findall('.//mods:mods/mods:identifier', namespaces=ns): 
    if re.match(call_num_pattern, identifier.text):
        print(identifier.text)
    else:
        print(f'invalid LCCN: { identifier.text }')

lcwaN0010234
invalid LCCN: 85999
invalid LCCN: 109353
lcwaN0001999
invalid LCCN: 91224
invalid LCCN: 109272
lcwaN0003238
invalid LCCN: 91275
invalid LCCN: 109273
invalid LCCN: 96782
lcwaN0010144
invalid LCCN: nan
lcwaN0010145
invalid LCCN: 82949
invalid LCCN: 109227
lcwaN0012178
invalid LCCN: 85778
lcwaN0012179
invalid LCCN: 85779
lcwaN0012180
invalid LCCN: 85780
lcwaN0012184
invalid LCCN: 85784
lcwaN0012195
invalid LCCN: 85795
lcwaN0010932
invalid LCCN: 110933
lcwaN0010933
invalid LCCN: 110934
lcwaN0010936
invalid LCCN: 110937
lcwaN0010937
invalid LCCN: 110938
lcwaN0010940
invalid LCCN: nan
lcwaN0010888
lcwaN0010226
lcwaN0009692
lcwaN0009700
lcwaN0010401
invalid LCCN: lcwaE0008846
invalid LCCN: lcwaE0008263
invalid LCCN: lcwaE0008338
invalid LCCN: lcwaE0008918
invalid LCCN: lcwaE0008001


In [46]:
#lxml insert attributes to make a more complete metadata record

xml_records = etree.parse(large_MODS_collec)

# regex pattern to identify the call number:
call_num_pattern = re.compile(r'[a-z]{4}N\d{7}')

for identifier in xml_records.findall('.//mods:mods/mods:identifier', namespaces=ns): 
    if re.match(call_num_pattern, identifier.text):
        print(identifier.text)
        identifier.attrib['displaylabel'] = 'Local Call Number'
        identifier.attrib['invalid'] = 'no'
        identifier.attrib['type'] = 'local_call_number'
        print('  ',identifier.attrib)

lcwaN0010234
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0001999
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0003238
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0010144
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0010145
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0012178
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0012179
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0012180
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0012184
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}
lcwaN0012195
   {'displaylabel': 'Local Call Number', 'invalid': 'no', 'type': 'local_call_number'}


#### Data validation - ADD new sample file TODO

7 - Data validation: check the reference IDs to ensure that they are in the proper format, then identify for correction as needed. 

In [ ]:
'''skip this cell TODO
#lxml
#filter the IDs, then do a regex match?

xml_records = etree.parse(MODS_Collection)

# previously we used a regex pattern to identify the call number:
# call_num_pattern = re.compile(r'[a-z]{4}N\d{7}')
# this time, be more specific and look for the lcwa string at the beginning:
call_num_pattern = re.compile(r'^blcwa[A-Z]{1}\d{7}')

for identifier in xml_records.findall('.//mods:mods/mods:identifier', namespaces=ns): 
    print(identifier.text)
    if re.match(call_num_pattern, identifier.text):
        print(identifier.tag, identifier.text, identifier.attrib)
'''

#### 8. Saving the Updated Metadata

Activity 8: Now, let's write the updated metadata to a new file. 

In [ ]:
#lxml - write out to a new file...

newfile = OUT / '2018_lcwa_MODS_25_updated.xml'

call_num_pattern = re.compile(r'[a-z]{4}N\d{7}')

xml_records = etree.parse(large_MODS_collec)

for mods in xml_records.findall('.//mods:mods', namespaces=ns):
    for identifier in mods.findall('.//mods:identifier', namespaces=ns):
        if re.match(call_num_pattern, identifier.text):
            # add attributes by assigning values
            identifier.set('type', 'local_call_number')
            identifier.set('invalid', 'no')
            identifier.set('displaylabel', 'Local Call Number')

#new file
xml_records.write(newfile, xml_declaration=True, encoding='utf-8', pretty_print=True)
print("Wrote a new file, you're welcome!")

:::{note} TODO: Activity 8 text
This cell now writes with `xml_records.write(...)` instead of `prettify()`. The text before the next cell ("Now, write the file with lxml, using proper namespace declarations and without prettify formatting") needs updating, since this cell is already lxml. Only the filename differs between the two write cells.
:::

Now, write the file with lxml, using proper namespace declarations and without prettify formatting:

In [ ]:
#lxml write file

newfile = OUT / '2018_lcwa_MODS_25_updatedlxml.xml'

# set up regex
call_num_pattern = re.compile(r'[a-z]{4}N\d{7}')

metadata = etree.parse(MODS_file)
xml_records = metadata.getroot()

for mods in xml_records.findall('mods:mods', namespaces=ns):
    for identifier in mods.find('.//mods:identifier', namespaces=ns):
        if re.match(call_num_pattern, identifier.text):
            # add attributes by assigning values
            identifier.set('type', 'local_call_number')
            identifier.set('invalid', 'no')
            identifier.set('displaylabel', 'Local Call Number')
            identifier.set('updated', 'true')

#new file
metadata.write(newfile, xml_declaration=True, encoding='utf-8', method='xml')
print("Wrote a new file, you're welcome!")

:::{note} TODO: Bugs in the Activity 8 lxml cell (not changed)
Two problems remain in this cell. (1) `mods.find('.//mods:identifier', ...)` returns a single element, so the `for` loop iterates that element's children, and the attribute-setting block most likely never runs. It should be `.findall()`. (2) It parses `MODS_file` (5 records) while the rest of the section uses `large_MODS_collec` (25 records). The `ET.register_namespace()` lines were removed because `xml.etree.ElementTree` is no longer imported and lxml does not use that registry.
:::